# Cache-representation identity benchmark

This notebook runs the checked-in multiprocess `ObjectKey` identity benchmark. It compares legacy and versioned key expansion, verifies that legacy key bytes remain unchanged, and proves that incompatible representations enter distinct namespaces. The measured path is CPU-side scheduler work; no GPU is required. Run all cells from the LMCache repository root.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
# Standard
from pathlib import Path
import json
import shlex
import subprocess
import sys

repo_candidates = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in repo_candidates if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("run from the LMCache repository")
script = repo_root / "benchmarks/microbenchmark/cache_identity_benchmark.py"
command = [
    sys.executable,
    str(script),
    "--chunks",
    "512",
    "--world-size",
    "8",
    "--object-groups",
    "4",
    "--iterations",
    "51",
    "--warmup",
    "7",
]
print("$", shlex.join(command))
completed = subprocess.run(
    command, check=False, capture_output=True, text=True, cwd=repo_root
)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"benchmark exited with {completed.returncode}")
print(completed.stdout)
evidence = json.loads(completed.stdout)
assert evidence["dimensions"]["object_keys_per_run"] == 16_384
assert all(evidence["invariants"].values())
print("Cache identity namespace and legacy-compatibility checks passed")